# Dissertation Scenario Generator Lab

Use this notebook to configure the synthetic scenario generator, generate JSON data, inspect model-ready objects, and visualise the base spread, manual baseline, and reference witness maintenance schedules.

The notebook imports `generate_dissertation_scenarios.py` from the same folder, so keep this notebook beside the generator file.

## 1. Imports

This notebook intentionally avoids pandas, matplotlib, or other third-party packages. It uses plain Python plus Jupyter's HTML display.

In [ ]:
from pathlib import Path
from html import escape
from IPython.display import HTML, display
import json
import random

import generate_dissertation_scenarios as gen

print(f"Loaded generator with {len(gen.OCCASION_NAMES)} occasions")
print(gen.OCCASION_NAMES)

## 2. Choose Scenario And Config

Edit this cell, then rerun the cells below. For one-shiploader topology, set `shiploader_count=1` and usually `shiploader_rate_tph=9166.0`.

In [ ]:
OCCASION = "priority_campaign_conflict"
OUTPUT_DIR = Path("generated_from_notebook")

CONFIG = gen.GeneratorConfig(
    horizon_days=90,
    seed=42,
    minimum_throughput_ratio=0.55,
    base_supply_tpd=178_000.0,
    base_demand_tpd=184_000.0,
    noise_fraction=0.035,
    initial_inventory_fill_ratio=0.48,
    shiploader_count=2,
    shiploader_rate_tph=4583.0,
)

CONFIG.validate()
print(CONFIG)

## 3. Build One Scenario

`scenario` is the rich Python object. `model_data` is what you would pass to an optimiser. It deliberately excludes the hidden reference schedule and design evidence.

In [ ]:
scenario = gen.build_scenario(OCCASION, CONFIG)
model_data = scenario.optimizer_objects()

baseline_metrics = gen.evaluate_schedule(scenario, scenario.manual_baseline)
reference_metrics = gen.evaluate_schedule(scenario, scenario.reference_schedule)

print("Scenario:", scenario.metadata["scenario_name"])
print("Description:", scenario.metadata["occasion_description"])
print("Expected feasibility:", scenario.metadata["expected_feasibility"])
print("Design checks pass:", scenario.design_evidence["occasion_checks"]["all_checks_pass"])
print("Model data keys:", len(model_data))

## 4. Generate JSON Files

For a single scenario, this writes one JSON file. To avoid accidental overwrites when changing seeds, the default filename includes the seed. Set `GENERATE_ALL = True` to generate every occasion for the current config.

In [ ]:
GENERATE_ALL = False
INCLUDE_SEED_IN_FILENAME = True

if GENERATE_ALL:
    # generate_suite writes standard names like balanced_control.json.
    # Use a seed-specific output directory to avoid overwriting another run.
    suite_dir = OUTPUT_DIR / f"seed_{CONFIG.seed:06d}"
    generated = gen.generate_suite(suite_dir, CONFIG)
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if INCLUDE_SEED_IN_FILENAME:
        filename = f"{OCCASION}_seed_{CONFIG.seed:06d}.json"
    else:
        filename = f"{OCCASION}.json"
    path = gen.write_scenario(scenario, OUTPUT_DIR / filename)
    generated = {OCCASION: path}

for name, path in generated.items():
    print(f"{name}: {path}")

## 4b. Batch Generate Many Seeds Without Overwriting

Use this when you want, for example, 100 independent instances of one occasion. Each seed is written to its own folder: `generated_seed_batch/<occasion>/seed_000042/<occasion>.json`.

In [ ]:
BATCH_GENERATE = False
BATCH_OCCASION = OCCASION
BATCH_OUTPUT_DIR = Path("generated_seed_batch")
SEEDS = range(42, 142)  # 100 seeds: 42 through 141

def config_with_seed(config, seed):
    return gen.GeneratorConfig(
        horizon_days=config.horizon_days,
        seed=seed,
        minimum_throughput_ratio=config.minimum_throughput_ratio,
        base_supply_tpd=config.base_supply_tpd,
        base_demand_tpd=config.base_demand_tpd,
        noise_fraction=config.noise_fraction,
        initial_inventory_fill_ratio=config.initial_inventory_fill_ratio,
        number_of_products=config.number_of_products,
        operating_hours_per_day=config.operating_hours_per_day,
        shiploader_count=config.shiploader_count,
        train_unloader_rate_tph=config.train_unloader_rate_tph,
        stacker_rate_tph=config.stacker_rate_tph,
        reclaimer_rate_tph=config.reclaimer_rate_tph,
        shiploader_rate_tph=config.shiploader_rate_tph,
        arc_secondary_rate_tph=config.arc_secondary_rate_tph,
        zone_capacity_tonnes=config.zone_capacity_tonnes,
    )

def generate_seed_batch(occasion, seeds, base_config, output_dir):
    written = []
    for seed in seeds:
        cfg = config_with_seed(base_config, int(seed))
        scenario_i = gen.build_scenario(occasion, cfg)
        seed_dir = output_dir / occasion / f"seed_{cfg.seed:06d}"
        path = gen.write_scenario(scenario_i, seed_dir / f"{occasion}.json")
        written.append(path)
    manifest = {
        "occasion": occasion,
        "seed_count": len(written),
        "seeds": [int(seed) for seed in seeds],
        "files": [str(path) for path in written],
        "shiploader_count": base_config.shiploader_count,
        "special_product_ids": ["X1"],
    }
    manifest_path = output_dir / occasion / "seed_batch_manifest.json"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    return written, manifest_path

if BATCH_GENERATE:
    written, manifest_path = generate_seed_batch(BATCH_OCCASION, SEEDS, CONFIG, BATCH_OUTPUT_DIR)
    print(f"Wrote {len(written)} scenarios")
    print(f"Manifest: {manifest_path}")
    print("First file:", written[0])
    print("Last file:", written[-1])
else:
    print("Set BATCH_GENERATE = True to write the seed batch.")

## 5. Inspect Products, X1 Requirements, And Model Keys

In [ ]:
def show_json(value, title=None):
    if title:
        print(title)
    print(json.dumps(gen._json_safe(value), indent=2))

show_json({p.product_id: {"name": p.name, "is_special": p.is_special, "share": p.share_of_volume, "blending_allowed": p.blending_allowed} for p in scenario.products}, "Products")
show_json(model_data["special_product_requirements"], "Special product requirements")

key_groups = [
    "SPECIAL_PRODUCTS",
    "product_is_special",
    "product_blending_allowed",
    "special_product_compatible_zones",
    "special_product_required_assets",
    "special_product_compatible_arcs",
    "zone_footprint_capacity_m2",
    "product_may_share_zone",
    "special_product_minimum_tpd",
    "special_product_campaign_minimum_tonnes",
]
print("\nSelected model-ready keys:")
for key in key_groups:
    print("-", key, "present =", key in model_data)

## 6. Inspect Daily Supply, Demand, And X1 Campaign Data

In [ ]:
def dense(records):
    out = {}
    for record in records:
        out.setdefault(record.day, {})[record.product_id] = record.tonnes
    return out

supply_by_day = dense(scenario.inbound_supply)
demand_by_day = dense(scenario.outbound_demand)

sample_days = list(range(0, min(5, len(scenario.days))))
if scenario.priority_activities:
    activity = scenario.priority_activities[0]
    sample_days += list(range(activity.start_day, min(activity.start_day + 3, activity.end_day)))
sample_days = sorted(set(sample_days))

rows = []
for day in sample_days:
    rows.append({
        "day": day,
        "total_supply": round(sum(supply_by_day[day].values()), 1),
        "X1_supply": round(supply_by_day[day].get("X1", 0.0), 1),
        "total_demand": round(sum(demand_by_day[day].values()), 1),
        "X1_demand": round(demand_by_day[day].get("X1", 0.0), 1),
        "X1_minimum_tpd": model_data.get("special_product_minimum_tpd", {}).get(("X1", day), 0.0),
    })

show_json(rows, "Sample daily rows")

## 7. Visualisation Helpers

These functions render compact HTML tables and timeline bars for schedules and metrics.

In [ ]:
STAGE_ORDER = {"train_unloader": 0, "stacker": 1, "reclaimer": 2, "shiploader": 3}

def build_base_spread_schedule(occasion, config):
    rng = random.Random(gen._stable_seed(config.seed, occasion))
    base = gen._build_base_scenario(config, rng, occasion)
    return list(base.manual_baseline)

def decision_map(schedule):
    return {decision.task_id: decision for decision in schedule}

def mode_duration(task, mode_id):
    return next(mode.duration_days for mode in task.modes if mode.mode_id == mode_id)

def html_bar(start, duration, horizon, css_class, label):
    left = 100.0 * start / horizon
    width = max(0.8, 100.0 * duration / horizon)
    title = escape(f"{label}: day {start}, {duration} days")
    return f'<span class="bar {css_class}" title="{title}" style="left:{left:.3f}%;width:{width:.3f}%"></span>'

def protected_window_spans(scenario):
    spans = []
    for activity in scenario.priority_activities:
        spans.append((
            activity.start_day - activity.pre_campaign_buffer_days,
            activity.end_day + activity.post_campaign_buffer_days,
            f"{activity.activity_id} protected window",
        ))
    return spans

def schedule_visualiser(scenario, base_schedule=None):
    horizon = len(scenario.days)
    base_schedule = base_schedule or build_base_spread_schedule(scenario.metadata["scenario_name"], CONFIG)
    maps = {
        "base": decision_map(base_schedule),
        "manual": decision_map(scenario.manual_baseline),
        "reference": decision_map(scenario.reference_schedule),
    }
    asset_stage = {asset.asset_id: asset.stage for asset in scenario.assets}
    tasks = sorted(scenario.maintenance_tasks, key=lambda t: (STAGE_ORDER[asset_stage[t.asset_id]], t.asset_id))
    spans = protected_window_spans(scenario)

    rows = []
    for task in tasks:
        lanes = []
        for lane_name in ("base", "manual", "reference"):
            decision = maps[lane_name][task.task_id]
            duration = mode_duration(task, decision.mode_id)
            protected = "".join(
                html_bar(max(0, start), min(horizon, end) - max(0, start), horizon, "protected", label)
                for start, end, label in spans
                if min(horizon, end) > max(0, start)
            )
            lanes.append(
                f'<div class="lane"><span class="lane-label">{lane_name}</span>{protected}'
                f'{html_bar(decision.start_day, duration, horizon, lane_name, decision.mode_id)}</div>'
            )
        rows.append(
            '<tr>'
            f'<td><code>{escape(task.task_id)}</code></td>'
            f'<td>{escape(task.asset_id)}<br><small>{escape(asset_stage[task.asset_id])}</small></td>'
            f'<td>{task.earliest_start_day}-{task.latest_start_day}<br><small>due {task.lifecycle_due_day}</small></td>'
            f'<td class="timeline-cell">{"".join(lanes)}</td>'
            '</tr>'
        )

    css = """
    <style>
    .schedule-wrap {font-family: Arial, sans-serif; color:#182230;}
    .legend {display:flex; gap:12px; flex-wrap:wrap; margin:8px 0 12px; font-size:12px;}
    .chip {display:inline-flex; align-items:center; gap:5px;}
    .swatch {width:13px; height:8px; border-radius:3px; display:inline-block;}
    table.schedule {border-collapse:collapse; width:100%; background:white;}
    table.schedule th, table.schedule td {border:1px solid #dfe5ec; padding:7px; vertical-align:middle; font-size:12px;}
    table.schedule th {background:#10233f; color:white; text-align:left;}
    .timeline-cell {min-width:520px;}
    .lane {position:relative; height:18px; margin:3px 0; background:repeating-linear-gradient(to right,#f4f7fa 0,#f4f7fa calc(100% / 9),#e5ebf2 calc(100% / 9),#e5ebf2 calc(100% / 9 + 1px)); border-radius:4px; overflow:hidden;}
    .lane-label {position:absolute; left:4px; top:2px; z-index:3; font-size:10px; color:#526174; background:rgba(255,255,255,.72); padding:0 3px; border-radius:3px;}
    .bar {position:absolute; top:3px; height:12px; border-radius:4px; z-index:2;}
    .bar.base {background:#2367b1; opacity:.80;}
    .bar.manual {background:#a56500; opacity:.88;}
    .bar.reference {background:#6d4fb1; opacity:.88;}
    .bar.protected {top:0; height:18px; background:#f3b54a; opacity:.22; z-index:1;}
    small {color:#667085;}
    </style>
    """
    html = css + '<div class="schedule-wrap">'
    html += f'<h3>{escape(scenario.metadata["scenario_name"])} maintenance schedules</h3>'
    html += '<div class="legend">'
    html += '<span class="chip"><span class="swatch" style="background:#2367b1"></span>base spread</span>'
    html += '<span class="chip"><span class="swatch" style="background:#a56500"></span>manual baseline</span>'
    html += '<span class="chip"><span class="swatch" style="background:#6d4fb1"></span>reference witness</span>'
    html += '<span class="chip"><span class="swatch" style="background:#f3b54a"></span>protected X1 window</span>'
    html += '</div>'
    html += '<table class="schedule"><thead><tr><th>Task</th><th>Asset</th><th>Window</th><th>Timeline, days 0 to {}</th></tr></thead><tbody>'.format(horizon - 1)
    html += ''.join(rows)
    html += '</tbody></table></div>'
    return HTML(html)

def metrics_table(baseline, reference):
    keys = [
        "total_throughput_tonnes",
        "unmet_demand_tonnes",
        "minimum_throughput_violation_days",
        "special_product_throughput_tonnes",
        "special_product_shortfall_tonnes",
        "special_product_campaign_volume_shortfall_tonnes",
        "special_product_footprint_violation_days",
        "special_product_maintenance_conflict_days",
        "objective_proxy",
    ]
    rows = []
    for key in keys:
        b = baseline.get(key, 0)
        r = reference.get(key, 0)
        delta = r - b if isinstance(b, (int, float)) and isinstance(r, (int, float)) else ""
        rows.append(f"<tr><td><code>{escape(key)}</code></td><td>{b}</td><td>{r}</td><td>{delta}</td></tr>")
    html = """
    <style>.metrics th,.metrics td{border:1px solid #dfe5ec;padding:7px;font-size:12px}.metrics{border-collapse:collapse;width:100%}.metrics th{background:#10233f;color:white;text-align:left}</style>
    <table class="metrics"><thead><tr><th>Metric</th><th>Manual baseline</th><th>Reference witness</th><th>Reference - baseline</th></tr></thead><tbody>
    """ + "".join(rows) + "</tbody></table>"
    return HTML(html)

## 8. Visualise Base, Manual Baseline, And Reference Schedules

The blue bars are the base spread schedule before the occasion overlay. Amber bars are the manual baseline used as the fixed comparator. Purple bars are the hidden reference witness. The pale amber band shows the protected X1 interval when the selected scenario has a campaign.

In [ ]:
base_schedule = build_base_spread_schedule(OCCASION, CONFIG)
display(schedule_visualiser(scenario, base_schedule))
display(metrics_table(baseline_metrics, reference_metrics))

## 9. Compare All Occasions Quickly

This is useful after changing `CONFIG` to confirm the benchmark suite still behaves as intended.

In [ ]:
summary = []
for name in gen.OCCASION_NAMES:
    s = gen.build_scenario(name, CONFIG)
    b = s.design_evidence["manual_baseline_metrics"]
    r = s.design_evidence["reference_schedule_metrics"]
    summary.append({
        "occasion": name,
        "checks_pass": s.design_evidence["occasion_checks"]["all_checks_pass"],
        "expected_feasibility": s.metadata["expected_feasibility"],
        "baseline_throughput": b["total_throughput_tonnes"],
        "reference_throughput": r["total_throughput_tonnes"],
        "baseline_X1_shortfall": b.get("special_product_shortfall_tonnes", 0),
        "reference_X1_shortfall": r.get("special_product_shortfall_tonnes", 0),
    })

show_json(summary, "Occasion summary")

## 10. Useful Experiments

- Change `OCCASION` to `balanced_control`, `yard_overflow_risk`, or `partial_shutdown_choice` and rerun from section 3.
- Change `shiploader_count` to `1` and `shiploader_rate_tph` to `9166.0` to inspect the one-shiploader topology.
- Increase `initial_inventory_fill_ratio` to stress the yard.
- Change `minimum_throughput_ratio` to make minimum-throughput constraints tighter or looser.
- Set `GENERATE_ALL = True` to write all scenario JSON files for the current config into a seed-specific folder.
- Set `BATCH_GENERATE = True` in section 4b to create many non-overwriting seed instances for one occasion.